# Generate Synthetic Credit Application Data with OpenAI

This notebook generates a complete synthetic credit application dataset using OpenAI's GPT-4o mini model. We'll create structured data, unstructured text descriptions, and default outcomes all through AI generation.

## Learning Objectives
- Use OpenAI API to generate realistic credit application data
- Create structured financial data through AI prompting
- Generate unstructured text descriptions for loan applications
- Determine default outcomes using AI reasoning
- Store all generated data in JSON format

## Setup and Imports

In [ ]:
!pip install openai pandas tiktoken
import os
import json
import openai
from openai import OpenAI
import pandas as pd
import numpy as np
from datetime import datetime
import time
import random
import tiktoken

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("Libraries imported and OpenAI client initialized successfully!")

DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


ModuleNotFoundError: No module named 'openai'

## Generate Structured Credit Application Data

First, let's define the parameters and distributions for our structured data:

In [ ]:
import random
import numpy as np
import json

# Define loan purposes and their typical characteristics
LOAN_PURPOSES = {
    'home_improvement': {'typical_amount_range': (5000, 50000), 'risk_factor': 0.8},
    'debt_consolidation': {'typical_amount_range': (3000, 30000), 'risk_factor': 1.2},
    'business': {'typical_amount_range': (10000, 100000), 'risk_factor': 1.5},
    'education': {'typical_amount_range': (2000, 25000), 'risk_factor': 0.7},
    'medical': {'typical_amount_range': (1000, 15000), 'risk_factor': 0.9},
    'vacation': {'typical_amount_range': (2000, 15000), 'risk_factor': 1.3},
    'wedding': {'typical_amount_range': (3000, 25000), 'risk_factor': 1.1},
    'car': {'typical_amount_range': (5000, 40000), 'risk_factor': 0.9},
    'other': {'typical_amount_range': (1000, 20000), 'risk_factor': 1.0}
}

CREDIT_HISTORY_LEVELS = ['excellent', 'good', 'fair', 'poor']

def generate_structured_data_with_ai(n_samples=100):
    """Generate structured credit application data using OpenAI"""
    
    prompt = f"""Generate {n_samples} realistic credit application records in JSON format. Each record should include:
    - applicant_id: Unique identifier (format: APP_XXXXXX)
    - age: Age between 18-80
    - income: Annual income in USD (realistic range)
    - loan_amount: Requested loan amount in USD
    - purpose: Loan purpose (home_improvement, debt_consolidation, business, education, medical, vacation, wedding, car, other)
    - credit_history: Credit rating (excellent, good, fair, poor)
    - employment_length: Years of employment (can be decimal)
    - debt_to_income: Debt-to-income ratio (0.0-1.0)
    - location: US state abbreviation
    - education: Education level (high_school, some_college, bachelors, masters, phd)
    
    Make the data realistic with appropriate correlations (e.g., higher income typically correlates with better credit history, older age with longer employment, etc.).
    
    Return ONLY a JSON array with no additional text or formatting."""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a financial data generator. Generate realistic credit application data in valid JSON format."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=4000
        )
        
        # Parse the JSON response
        data = json.loads(response.choices[0].message.content)
        return data
        
    except Exception as e:
        print(f"Error generating structured data: {e}")
        return None

# Generate structured data
print("Generating structured credit application data...")
structured_data = generate_structured_data_with_ai(100)

if structured_data:
    print(f"Successfully generated {len(structured_data)} structured records")
    print("Sample record:")
    print(json.dumps(structured_data[0], indent=2))
else:
    print("Failed to generate structured data")

## Generate Unstructured Text Descriptions

Now let's use OpenAI to generate realistic loan application text descriptions:

In [ ]:
def generate_text_descriptions_with_ai(structured_records):
    """Generate text descriptions for each credit application using OpenAI"""
    
    enhanced_records = []
    
    for i, record in enumerate(structured_records):
        # Create a detailed prompt for text generation
        prompt = f"""Write a realistic loan application description for this applicant:
        
        Age: {record['age']}
        Income: ${record['income']:,}
        Loan Amount: ${record['loan_amount']:,}
        Purpose: {record['purpose']}
        Credit History: {record['credit_history']}
        Employment Length: {record['employment_length']} years
        Debt-to-Income: {record['debt_to_income']:.2f}
        Education: {record['education']}
        Location: {record['location']}
        
        Generate a 2-3 sentence description that sounds like a real loan application explanation. Include:
        - Why they need the loan
        - Brief mention of their financial situation
        - Confidence in repayment ability
        
        Make it sound natural and personalized based on their characteristics."""
        
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are writing loan application descriptions. Write realistic, personalized descriptions based on applicant characteristics."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.8,
                max_tokens=200
            )
            
            # Add text description to record
            enhanced_record = record.copy()
            enhanced_record['text_description'] = response.choices[0].message.content.strip()
            enhanced_records.append(enhanced_record)
            
            # Rate limiting
            time.sleep(0.1)
            
            if (i + 1) % 10 == 0:
                print(f"Generated {i + 1} text descriptions...")
                
        except Exception as e:
            print(f"Error generating text for record {i}: {e}")
            # Add record without text description
            enhanced_record = record.copy()
            enhanced_record['text_description'] = "Standard loan application request."
            enhanced_records.append(enhanced_record)
    
    return enhanced_records

# Generate text descriptions
print("Generating text descriptions for loan applications...")
enhanced_data = generate_text_descriptions_with_ai(structured_data)

print(f"Successfully generated text descriptions for {len(enhanced_data)} records")
print("\nSample text description:")
print(f"Purpose: {enhanced_data[0]['purpose']}")
print(f"Description: {enhanced_data[0]['text_description']}")

## Generate Default Outcomes Using AI

Now let's use OpenAI to determine default outcomes based on all available information:

In [ ]:
def get_token_ids(text, model="gpt-4o-mini"):
    """Get token IDs for given text"""
    # Use the appropriate encoding for the model
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    return encoding.encode(text)

def generate_default_outcomes_with_token_probs(records):
    """Generate default outcomes using token probabilities for D and ND tokens"""
    
    # Get token IDs for D and ND
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    d_token_id = encoding.encode("D")[0]  # Default token
    nd_token_id = encoding.encode("ND")[0]  # No Default token
    
    print(f"Token IDs: D={d_token_id}, ND={nd_token_id}")
    
    final_records = []
    
    for i, record in enumerate(records):
        # Create prompt for binary classification
        prompt = f"""Based on the following credit application, will this customer default on their loan?
        
        Customer Profile:
        - Age: {record['age']}
        - Annual Income: ${record['income']:,}
        - Loan Amount: ${record['loan_amount']:,}
        - Purpose: {record['purpose']}
        - Credit History: {record['credit_history']}
        - Employment Length: {record['employment_length']} years
        - Debt-to-Income Ratio: {record['debt_to_income']:.2f}
        - Education: {record['education']}
        - Location: {record['location']}
        - Application Description: {record['text_description']}
        
        Consider standard credit risk factors:
        - Credit history quality
        - Debt-to-income ratio
        - Employment stability
        - Income level relative to loan amount
        - Loan purpose risk
        
        Respond with only one token: D (for default) or ND (for no default)."""
        
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a credit risk analyst. Respond with only 'D' for default or 'ND' for no default."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=1,
                logprobs=True,
                top_logprobs=10
            )
            
            # Get the response token and its logprobs
            choice = response.choices[0]
            
            if choice.logprobs and choice.logprobs.content:
                # Get the first token's logprobs
                token_logprobs = choice.logprobs.content[0]
                
                # Extract probabilities for D and ND tokens
                d_logprob = None
                nd_logprob = None
                
                # Check the top logprobs for D and ND tokens
                for top_logprob in token_logprobs.top_logprobs:
                    if top_logprob.token == "D":
                        d_logprob = top_logprob.logprob
                    elif top_logprob.token == "ND":
                        nd_logprob = top_logprob.logprob
                
                # If we don't find both tokens in top logprobs, use the predicted token
                predicted_token = choice.message.content.strip()
                if predicted_token == "D" and d_logprob is None:
                    d_logprob = token_logprobs.logprob
                elif predicted_token == "ND" and nd_logprob is None:
                    nd_logprob = token_logprobs.logprob
                
                # Convert logprobs to probabilities
                if d_logprob is not None and nd_logprob is not None:
                    d_prob = np.exp(d_logprob)
                    nd_prob = np.exp(nd_logprob)
                    
                    # Normalize probabilities
                    total_prob = d_prob + nd_prob
                    d_prob_normalized = d_prob / total_prob
                    nd_prob_normalized = nd_prob / total_prob
                    
                    default_probability = d_prob_normalized
                    no_default_probability = nd_prob_normalized
                    
                elif predicted_token == "D":
                    default_probability = 0.9  # High confidence if only D token found
                    no_default_probability = 0.1
                elif predicted_token == "ND":
                    default_probability = 0.1  # Low confidence if only ND token found
                    no_default_probability = 0.9
                else:
                    # Fallback if neither token is found
                    default_probability = 0.5
                    no_default_probability = 0.5
                
                # Determine binary outcome based on probability
                default_outcome = 1 if default_probability > 0.5 else 0
                
            else:
                # Fallback if no logprobs available
                predicted_token = choice.message.content.strip()
                if predicted_token == "D":
                    default_probability = 0.9
                    no_default_probability = 0.1
                    default_outcome = 1
                else:
                    default_probability = 0.1
                    no_default_probability = 0.9
                    default_outcome = 0
            
            # Add results to record
            enhanced_record = record.copy()
            enhanced_record['default_probability'] = round(default_probability, 4)
            enhanced_record['no_default_probability'] = round(no_default_probability, 4)
            enhanced_record['default_outcome'] = default_outcome
            enhanced_record['predicted_token'] = predicted_token if 'predicted_token' in locals() else choice.message.content.strip()
            
            final_records.append(enhanced_record)
            
            # Rate limiting
            time.sleep(0.1)
            
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(records)} records")
                
        except Exception as e:
            print(f"Error processing record {i}: {e}")
            # Add record with default values
            enhanced_record = record.copy()
            enhanced_record['default_probability'] = 0.1
            enhanced_record['no_default_probability'] = 0.9
            enhanced_record['default_outcome'] = 0
            enhanced_record['predicted_token'] = "ND"
            final_records.append(enhanced_record)
    
    return final_records

# Generate default outcomes using token probabilities
print("Generating default outcomes using token probabilities...")
complete_data = generate_default_outcomes_with_token_probs(enhanced_data)

print(f"Successfully generated token-based risk assessments for {len(complete_data)} records")

# Display statistics
default_probs = [record['default_probability'] for record in complete_data]
default_outcomes = [record['default_outcome'] for record in complete_data]
predicted_tokens = [record['predicted_token'] for record in complete_data]

print(f"\nDefault Statistics:")
print(f"Average default probability: {np.mean(default_probs):.3f}")
print(f"Actual default rate: {np.mean(default_outcomes):.3f}")
print(f"Min/Max default probability: {np.min(default_probs):.3f} / {np.max(default_probs):.3f}")
print(f"Token predictions: D={predicted_tokens.count('D')}, ND={predicted_tokens.count('ND')}")

# Show sample with token probabilities
print(f"\nSample record with token probabilities:")
sample = complete_data[0]
print(f"Applicant: {sample['applicant_id']}")
print(f"Credit History: {sample['credit_history']}")
print(f"Predicted Token: {sample['predicted_token']}")
print(f"Default Probability: {sample['default_probability']:.3f}")
print(f"No Default Probability: {sample['no_default_probability']:.3f}")
print(f"Final Outcome: {'Default' if sample['default_outcome'] else 'No Default'}")

## Save Dataset to JSON

Let's save our complete dataset to a JSON file for use in subsequent notebooks:

In [ ]:
# Add metadata to the dataset
dataset_metadata = {
    "dataset_info": {
        "creation_date": datetime.now().isoformat(),
        "total_records": len(final_data),
        "generation_method": "OpenAI GPT-4o mini",
        "version": "1.0",
        "description": "Synthetic credit application dataset with structured data, text descriptions, and default outcomes"
    },
    "feature_descriptions": {
        "applicant_id": "Unique identifier for each applicant",
        "age": "Age of applicant in years",
        "income": "Annual income in USD",
        "loan_amount": "Requested loan amount in USD",
        "purpose": "Purpose of the loan",
        "credit_history": "Credit history rating (excellent, good, fair, poor)",
        "employment_length": "Years of employment",
        "debt_to_income": "Debt-to-income ratio (0-1)",
        "text_description": "Applicant's loan application narrative",
        "default_probability": "Probability of default (0-1)",
        "risk_rating": "Risk assessment (Low, Medium, High)",
        "risk_explanation": "Explanation of key risk factors",
        "default_label": "Binary outcome (0=no default, 1=default)"
    },
    "statistics": {
        "total_applicants": len(final_data),
        "default_rate": df_temp['default_label'].mean(),
        "avg_default_probability": df_temp['default_probability'].mean(),
        "risk_rating_distribution": df_temp['risk_rating'].value_counts().to_dict()
    }
}

# Complete dataset structure
complete_dataset = {
    "metadata": dataset_metadata,
    "applications": final_data
}

# Save the complete dataset to JSON
output_file = 'credit_applications_dataset.json'

# Add metadata
dataset = {
    'metadata': {
        'generated_date': datetime.now().isoformat(),
        'total_records': len(complete_data),
        'generation_method': 'OpenAI GPT-4o mini with token probabilities',
        'average_default_probability': np.mean(default_probs),
        'actual_default_rate': np.mean(default_outcomes),
        'prediction_method': 'Token probability for D (Default) and ND (No Default) tokens',
        'token_distribution': {
            'D_predictions': predicted_tokens.count('D'),
            'ND_predictions': predicted_tokens.count('ND')
        }
    },
    'feature_descriptions': {
        'applicant_id': 'Unique identifier for each applicant',
        'age': 'Age of applicant in years',
        'income': 'Annual income in USD',
        'loan_amount': 'Requested loan amount in USD',
        'purpose': 'Purpose of the loan',
        'credit_history': 'Credit history rating (excellent, good, fair, poor)',
        'employment_length': 'Years of employment',
        'debt_to_income': 'Debt-to-income ratio (0-1)',
        'location': 'US state abbreviation',
        'education': 'Education level',
        'text_description': 'Applicant loan application narrative',
        'default_probability': 'Normalized probability of default from D token',
        'no_default_probability': 'Normalized probability of no default from ND token',
        'default_outcome': 'Binary outcome (0=no default, 1=default)',
        'predicted_token': 'Token predicted by model (D or ND)'
    },
    'data': complete_data
}

# Save to JSON file
with open(output_file, 'w') as f:
    json.dump(dataset, f, indent=2)

print(f"Dataset saved to {output_file}")
print(f"Total records: {len(complete_data)}")
print(f"File size: {os.path.getsize(output_file)} bytes")

# Also save a simplified CSV for quick analysis
df_simple = pd.DataFrame(final_data)
df_simple.to_csv("credit_applications_simple.csv", index=False)
print(f"✅ Simplified dataset saved to credit_applications_simple.csv")

# Display sample records
print("\nSample records from final dataset:")
for i in range(2):
    print(f"\n--- Applicant {final_data[i]['applicant_id']} ---")
    print(f"Age: {final_data[i]['age']}, Income: ${final_data[i]['income']:,}")
    print(f"Loan: ${final_data[i]['loan_amount']:,} for {final_data[i]['purpose']}")
    print(f"Credit: {final_data[i]['credit_history']}")
    print(f"Description: {final_data[i]['text_description']}")
    print(f"Risk: {final_data[i]['risk_rating']} (p={final_data[i]['default_probability']:.3f})")
    print(f"Outcome: {'Default' if final_data[i]['default_label'] else 'No Default'}")

# Display sample complete record
print("\nSample complete record:")
sample_record = complete_data[0]
for key, value in sample_record.items():
    if key == 'text_description':
        print(f"{key}: {value[:100]}...")
    else:
        print(f"{key}: {value}")

## Validate Generated Data

Let's perform a quick validation of our generated dataset:

In [ ]:
# Load and validate the saved dataset
with open(output_file, 'r') as f:
    loaded_dataset = json.load(f)

print("Dataset validation:")
print(f"Metadata: {loaded_dataset['metadata']}")
print(f"Number of records: {len(loaded_dataset['data'])}")

# Convert to DataFrame for analysis
df = pd.DataFrame(loaded_dataset['data'])
print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Basic statistics
print(f"\nNumerical statistics:")
numerical_cols = ['age', 'income', 'loan_amount', 'employment_length', 'debt_to_income', 'default_probability']
print(df[numerical_cols].describe())

# Categorical distributions
print(f"\nCategorical distributions:")
categorical_cols = ['purpose', 'credit_history', 'education', 'location']
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().head())

# Default analysis
print(f"\nDefault analysis:")
print(f"Default rate by credit history:")
print(df.groupby('credit_history')['default_outcome'].mean().sort_values(ascending=False))

print(f"\nDataset generation completed successfully!")